This script identifies statistically meaningful increases in:
1. Gender equality (% female athletes) in Olympic delegations
2. Paralympic participation growth (total delegation size)
3. Gender equality within Paralympic delegations (intersectional analysis)

Methodology:
- We calculate edition-to-edition changes in % female athletes (percentage points)
- We filter for delegations ≥ 10 athletes to avoid small-sample noise
- We restrict to consecutive editions (≤ 8 year gap) to ensure comparability
- We identify both single-edition spikes AND sustained multi-edition trends
- Summer and Winter Games are analyzed separately (4-year cycles)

Data sources:
- Olympic athletes dataset (1896–2026): individual-level records
- Paralympic delegation dataset (1960–2026): country-level aggregates

#Setup

##Imports

In [1]:
pip install plotly kaleido

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 3.6 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

##Data loading

In [3]:
olympics = pd.read_excel('/content/Olympic Athletes.xlsx')
paralympics =pd.read_excel('/content/Paralympic Athletes.xlsx')

##Initial configuration

In [4]:
MIN_DELEGATION = 10      # Minimum total athletes to include a delegation
MAX_YEAR_GAP = 8         # Maximum years between editions for comparison

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Visual style
sns.set_theme(style="whitegrid", font_scale=1.1)
COLORS = {
    'Africa': '#E6A532',
    'Americas': '#2E86AB',
    'Asia': '#D64045',
    'Europe': '#1B4332',
    'Oceania': '#7209B7',
    'highlight': '#D64045',
    'background_line': '#CCCCCC',
    'female': '#E63946',
    'male': '#457B9D'}

In [5]:
# Deduplicate: one athlete can appear in multiple events per Games
olympics = olympics.drop_duplicates(subset=['Name', 'NOC', 'Year', 'Season', 'Gender'])
print(f"Unique athlete-edition records: {len(olympics):,}")

# Pivot: count Male/Female per NOC-Year-Season
olympics = (olympics
            .groupby(['NOC', 'Year', 'Season', 'Gender'])
            .size()
            .unstack(fill_value=0)
            .reset_index())
olympics.columns.name = None

for col in ['Female', 'Male']:
    if col not in olympics.columns:
        olympics[col] = 0

olympics['Total'] = olympics['Female'] + olympics['Male']
olympics['Pct_Female'] = (olympics['Female'] / olympics['Total'] * 100).round(2)
olympics['Type'] = 'Olympic'

print(f"Olympic delegations (NOC × Edition): {len(olympics):,}")
print(olympics.head())

Unique athlete-edition records: 212,997
Olympic delegations (NOC × Edition): 4,342
   NOC  Year  Season  Female  Male  Total  Pct_Female     Type
0  AFG  1936  Summer       0    15     15         0.0  Olympic
1  AFG  1948  Summer       0    25     25         0.0  Olympic
2  AFG  1956  Summer       0    12     12         0.0  Olympic
3  AFG  1960  Summer       0    12     12         0.0  Olympic
4  AFG  1964  Summer       0     8      8         0.0  Olympic


In [6]:
nagano_mask = (paralympics['Year'] == 1988) & (paralympics['Host_City'] == 'Nagano')
n_fixed = nagano_mask.sum()
paralympics.loc[nagano_mask, 'Year'] = 1998
print(f"  Fixed {n_fixed} Nagano records: 1988 → 1998")

# Map Season by (Year, Host_City)
WINTER_EDITIONS = {
    (1976, 'Ornskoldsvik'), (1980, 'Geilo'), (1984, 'Innsbruck'),
    (1988, 'Innsbruck'), (1992, 'Albertville, Tignes'), (1994, 'Lillehammer'),
    (1998, 'Nagano'), (2002, 'Salt Lake City'), (2006, 'Torino'),
    (2010, 'Vancouver'), (2014, 'Sochi'), (2018, 'Pyeongchang'),
    (2022, 'Beijing'), (2026, 'Milano-Cortina')}

paralympics['Season'] = paralympics.apply(
    lambda r: 'Winter' if (r['Year'], r['Host_City']) in WINTER_EDITIONS else 'Summer',
    axis=1)

# Aggregate by NOC-Year-Season (some years have multiple host cities)
paralympics = (paralympics.dropna(subset=['Country_Code'])
               .groupby(['Country_Code', 'Year', 'Season'])
               .agg(Male=('Men', 'sum'), Female=('Women', 'sum'), Total=('Total', 'sum'))
               .reset_index()
               .rename(columns={'Country_Code': 'NOC'}))

paralympics['Pct_Female'] = (paralympics['Female'] / paralympics['Total'] * 100).round(2)
paralympics['Type'] = 'Paralympic'

print(f"  Paralympic delegations (NOC × Edition): {len(paralympics):,}")

  Fixed 31 Nagano records: 1988 → 1998
  Paralympic delegations (NOC × Edition): 2,027


IOC Regions

In [7]:
IOC_REGIONS = {
    'Africa': ['ALG','ANG','BEN','BOT','BUR','BDI','CMR','CPV','CAF','CHA','COM','CGO','COD', 'CIV','DJI', 'EGY','ERI','SWZ','ETH','GAB','GAM',
               'GHA','GUI','GBS','GEQ','KEN','LES','LBR','LBA','MAD','MAW','MLI','MTN','MRI','MAR','MOZ','NAM','NIG','NGR','RWA','STP','SEN',
               'SEY','SLE','SOM','RSA','SSD','SUD','TAN','TOG','TUN','UGA', 'ZAM','ZIM'],
    'Americas': ['ANT','ARG','ARU','BAH','BAR','BIZ','BER','BOL','BRA','IVB','CAN','CAY','CHI', 'COL','CRC','CUB','DMA','DOM','ECU','ESA','GRN',
                 'GUA','GUY','HAI','HON','JAM', 'MEX','NCA','PAN','PAR','PER','PUR','SKN','LCA','VIN','SUR','TTO','URU','USA', 'VEN','ISV'],
    'Asia': ['AFG','BRN','BAN','BHU','BRU','CAM','CHN','TPE','GUM','HKG','IND','INA','IRQ','IRI','JOR','JPN','KAZ','KGZ','KOR','PRK','KUW','LAO',
             'LBN','MAS','MDV','MGL', 'MYA','NEP','OMA','PAK','PLE','PHI','QAT','KSA','SGP','SRI','SYR','TJK','THA','TLS','TKM','UAE','UZB',
             'VIE','YEM','MAC'],
    'Europe': ['ALB','AND','ARM','AUT','AZE','BLR','BEL','BIH','BUL','CRO','CYP','CZE','DEN', 'EST','FIN','FRA','GEO','GER','GBR','GRE','HUN',
               'ISL','IRL','ISR','ITA','KOS', 'LAT','LIE','LTU','LUX','MLT','MDA','MON','MNE','NED','MKD','NOR','POL','POR', 'ROU','RUS','SMR',
               'SRB','SVK','SLO','ESP','SWE','SUI','TUR','UKR','SCG','YUG', 'TCH','FRG','GDR','EUN','URS','ROC','AIN'],
    'Oceania': ['ASA','AUS','COK','FIJ','FSM','GUM','KIR','MHL','NRU','NZL','PLW','PNG','SAM', 'SOL','TGA','TUV','VAN']}

noc_to_region = {}
for region, nocs in IOC_REGIONS.items():
    for noc in nocs:
        noc_to_region[noc] = region

olympics['Region'] = olympics['NOC'].map(noc_to_region)
paralympics['Region'] = paralympics['NOC'].map(noc_to_region)

unmapped_oly = sorted(olympics[olympics.Region.isna()]['NOC'].unique())
unmapped_para = sorted(paralympics[paralympics.Region.isna()]['NOC'].unique())
print(f"  Unmapped Olympic NOCs (historical/special): {unmapped_oly}")
print(f"  Unmapped Paralympic NOCs: {unmapped_para}")

  Unmapped Olympic NOCs (historical/special): ['AHO', 'ANZ', 'BOH', 'CRT', 'EOR', 'IOA', 'LIB', 'MAL', 'NBO', 'NFL', 'RHO', 'SAA', 'UAR', 'UNK', 'VNM', 'WIF', 'YAR', 'YMD']
  Unmapped Paralympic NOCs: ['FRO', 'IPA', 'IPP', 'NPA', 'RPC', 'RPT', 'Refugees']


#1. Peak Identification

##1.1. Methodology

In [8]:
def calculate_edition_changes(df, value_col='Pct_Female', min_total=MIN_DELEGATION,
                               max_gap=MAX_YEAR_GAP, min_prev_total=MIN_DELEGATION):
    """
    Calculate the change in a metric between consecutive editions
    for each NOC within the same Season.

    Parameters
    ----------
    df : DataFrame with columns [NOC, Year, Season, Total, Region, <value_col>]
    value_col : column to track changes in (default: 'Pct_Female')
    min_total : minimum delegation size for current edition
    min_prev_total : minimum delegation size for previous edition
    max_gap : maximum year gap between editions

    Returns
    -------
    DataFrame with added columns: Prev_{value_col}, Delta_{value_col},
    Prev_Year, Prev_Total, Year_Gap
    """
    df_f = df[df['Total'] >= min_total].copy()
    df_f = df_f.sort_values(['NOC', 'Season', 'Year'])

    # Shift within NOC + Season groups
    grp = df_f.groupby(['NOC', 'Season'])
    df_f[f'Prev_{value_col}'] = grp[value_col].shift(1)
    df_f['Prev_Year'] = grp['Year'].shift(1)
    df_f['Prev_Total'] = grp['Total'].shift(1)

    # Calculate change
    df_f[f'Delta_{value_col}'] = df_f[value_col] - df_f[f'Prev_{value_col}']
    df_f['Year_Gap'] = df_f['Year'] - df_f['Prev_Year']

    # Filter for valid comparisons
    mask = (
        df_f[f'Delta_{value_col}'].notna() &
        (df_f['Year_Gap'] <= max_gap) &
        (df_f['Prev_Total'] >= min_prev_total))

    return df_f, df_f[mask].copy()


def find_sustained_increases(changes_df, value_col='Pct_Female', min_streak=3):
    """
    Find countries with N or more consecutive editions of increase
    in the given metric.

    Returns DataFrame with: NOC, Region, Consecutive_Increases,
    Period_Start, Period_End, Total_Delta_pp
    """
    delta_col = f'Delta_{value_col}'
    df = changes_df.sort_values(['NOC', 'Year']).copy()

    results = []
    for (noc, season), grp in df.groupby(['NOC', 'Season']):
        streak = 0
        max_streak = 0
        start_year = None
        best = {'start': None, 'end': None, 'delta': 0}
        cumulative_delta = 0

        for _, row in grp.iterrows():
            if row[delta_col] > 0:
                if streak == 0:
                    start_year = row['Year']
                    cumulative_delta = 0
                streak += 1
                cumulative_delta += row[delta_col]
                if streak > max_streak:
                    max_streak = streak
                    best = {
                        'start': start_year,
                        'end': row['Year'],
                        'delta': cumulative_delta}
            else:
                streak = 0
                cumulative_delta = 0

        if max_streak >= min_streak:
            results.append({
                'NOC': noc,
                'Season': season,
                'Region': grp['Region'].iloc[0],
                'Consecutive_Increases': max_streak,
                'Period_Start': int(best['start']),
                'Period_End': int(best['end']),
                'Total_Delta_pp': round(best['delta'], 1)})

    return (pd.DataFrame(results)
            .sort_values('Consecutive_Increases', ascending=False)
            .reset_index(drop=True))

##1.2. Olympic gender equality

In [9]:
print("--- Olympic Gender Equality (% Female) ---")
olympics_full, olympics_changes = calculate_edition_changes(olympics)
print(f"Valid comparisons: {len(olympics_changes):,}")

olympics_sustained = find_sustained_increases(olympics_changes)
print(f"Countries with 3+ consecutive increases: {len(olympics_sustained)}")

# Top 15 single-edition spikes (Summer)
DISPLAY_COLS_GENDER = [
    'NOC', 'Region', 'Year', 'Season', 'Female', 'Male', 'Total',
    'Pct_Female', 'Prev_Pct_Female', 'Delta_Pct_Female', 'Year_Gap']

print("\n📊 Top 15 single-edition increases in % Female (Summer)")
top_olympics_summer = (olympics_changes[olympics_changes.Season == 'Summer']
                       .nlargest(15, 'Delta_Pct_Female')[DISPLAY_COLS_GENDER])
display(top_olympics_summer)

# Sustained trends
print("\n📊 Sustained increases (3+ consecutive editions, Summer)")
olympics_sus_summer = olympics_sustained[olympics_sustained.Season == 'Summer']
display(olympics_sus_summer.head(15))

--- Olympic Gender Equality (% Female) ---
Valid comparisons: 2,075
Countries with 3+ consecutive increases: 95

📊 Top 15 single-edition increases in % Female (Summer)


,NOC,Region,Year,Season,Female,Male,Total,Pct_Female,Prev_Pct_Female,Delta_Pct_Female,Year_Gap
3132,PER,Americas,1976,Summer,13,0,13,100.00,15.00,85.00,4.0
870,COD,Africa,1996,Summer,12,2,14,85.71,11.76,73.95,4.0
3471,SEN,Africa,2000,Summer,19,7,26,73.08,0.00,73.08,4.0
2707,MLI,Africa,2008,Summer,14,3,17,82.35,9.52,72.83,4.0
839,CIV,Africa,1988,Summer,17,11,28,60.71,6.67,54.04,4.0
3477,SEN,Africa,2016,Summer,16,6,22,72.73,22.58,50.15,4.0
848,CIV,Africa,2024,Summer,9,4,13,69.23,19.35,49.88,4.0
846,CIV,Africa,2016,Summer,7,5,12,58.33,9.52,48.81,8.0
862,CMR,Africa,2012,Summer,22,10,32,68.75,21.88,46.87,4.0
586,BRN,Asia,2012,Summer,8,4,12,66.67,21.43,45.24,4.0



📊 Sustained increases (3+ consecutive editions, Summer)


,NOC,Season,Region,Consecutive_Increases,Period_Start,Period_End,Total_Delta_pp
0,GBR,Summer,Europe,11,1960,2000,26.9
1,BRA,Summer,Americas,8,1976,2004,45.3
2,ITA,Summer,Europe,8,1992,2020,32.3
3,FRA,Summer,Europe,7,1976,2000,24.0
4,USA,Summer,Americas,7,1996,2020,18.9
5,MAS,Summer,Asia,7,1996,2020,60.0
6,ALG,Summer,Africa,7,1988,2012,44.7
7,MGL,Summer,Asia,6,1976,2000,34.9
8,ECU,Summer,Americas,6,2000,2020,53.3
9,GER,Summer,Europe,6,1996,2016,11.2


In [10]:
regions = ['Africa', 'Americas', 'Asia', 'Europe', 'Oceania']
olympics_summer = olympics[(olympics['Season'] == 'Summer') & (olympics['Total'] >= MIN_DELEGATION)]

fig = make_subplots(rows=3, cols=2,
                    subplot_titles=regions + ['All Regions Compared'],
                    vertical_spacing=0.08, horizontal_spacing=0.06)

for idx, region in enumerate(regions):
    row = idx // 2 + 1
    col = idx % 2 + 1
    region_data = olympics_summer[olympics_summer['Region'] == region]

    # Each country as its own trace
    for noc in sorted(region_data['NOC'].unique()):
        country = region_data[region_data['NOC'] == noc].sort_values('Year')
        fig.add_trace(go.Scatter(
            x=country['Year'], y=country['Pct_Female'],
            mode='lines', name=noc,
            line=dict(color=COLORS['background_line'], width=1),
            opacity=0.4,
            hovertemplate=f'<b>{noc}</b><br>Year: %{{x}}<br>% Female: %{{y:.1f}}%<extra></extra>',
            legendgroup=region, showlegend=False), row=row, col=col)

    # Regional weighted average
    regional_avg = (region_data.groupby('Year')
                    .apply(lambda g: (g['Female'].sum() / g['Total'].sum() * 100))
                    .reset_index(name='Pct_Female'))
    fig.add_trace(go.Scatter(
        x=regional_avg['Year'], y=regional_avg['Pct_Female'],
        mode='lines+markers', name=f'{region} avg',
        line=dict(color=COLORS[region], width=3),
        marker=dict(size=5),
        hovertemplate=f'<b>{region} average</b><br>Year: %{{x}}<br>% Female: %{{y:.1f}}%<extra></extra>',
        legendgroup='averages', showlegend=True), row=row, col=col)

    # Parity line
    fig.add_hline(y=50, line_dash="dash", line_color="gray",
                  opacity=0.4, row=row, col=col)

# Summary subplot (row 3, col 2): all averages together
for region in regions:
    region_data = olympics_summer[olympics_summer['Region'] == region]
    regional_avg = (region_data.groupby('Year')
                    .apply(lambda g: (g['Female'].sum() / g['Total'].sum() * 100))
                    .reset_index(name='Pct_Female'))
    fig.add_trace(go.Scatter(
        x=regional_avg['Year'], y=regional_avg['Pct_Female'],
        mode='lines+markers', name=region,
        line=dict(color=COLORS[region], width=3),
        marker=dict(size=5),
        hovertemplate=f'<b>{region}</b><br>Year: %{{x}}<br>% Female: %{{y:.1f}}%<extra></extra>',
        legendgroup='summary', showlegend=False), row=3, col=2)
fig.add_hline(y=50, line_dash="dash", line_color="gray", opacity=0.4, row=3, col=2)

fig.update_yaxes(range=[0, 105], ticksuffix='%')
fig.update_layout(
    height=900, width=1100,
    title_text='Olympic Summer Games — Female Athlete Participation by IOC Region',
    title_font_size=16,
    hovermode='closest',
    template='plotly_white')
fig.show()

##1.3. Paralympic participation

In [11]:
print("--- Paralympic Participation (Total delegation) ---")
paralympics_full_total, paralympics_changes_total = calculate_edition_changes(
    paralympics, value_col='Total', min_total=10, min_prev_total=5)
print(f"Valid comparisons: {len(paralympics_changes_total):,}")

paralympics_sustained_total = find_sustained_increases(
    paralympics_changes_total, value_col='Total')
print(f"Countries with 3+ consecutive increases: {len(paralympics_sustained_total)}")

DISPLAY_COLS_TOTAL = [
    'NOC', 'Region', 'Year', 'Season', 'Total', 'Prev_Total',
    'Delta_Total', 'Year_Gap']

print("\n📊 Top 15 absolute growth in delegation size (Summer)")
top_paralympics_growth = (paralympics_changes_total[paralympics_changes_total.Season == 'Summer']
                          .nlargest(15, 'Delta_Total')[DISPLAY_COLS_TOTAL])
display(top_paralympics_growth)

print("\n📊 Sustained delegation growth (3+ consecutive, Summer)")
paralympics_sus_total = paralympics_sustained_total[paralympics_sustained_total.Season == 'Summer']
display(paralympics_sus_total.head(15))

--- Paralympic Participation (Total delegation) ---
Valid comparisons: 806
Countries with 3+ consecutive increases: 37

📊 Top 15 absolute growth in delegation size (Summer)


,NOC,Region,Year,Season,Total,Prev_Total,Delta_Total,Year_Gap
1088,KOR,Asia,1988,Summer,226,18.0,208.0,4.0
550,ESP,Europe,1992,Summer,233,50.0,183.0,4.0
1941,USA,Americas,1988,Summer,371,236.0,135.0,4.0
366,CHN,Asia,2008,Summer,331,197.0,134.0,4.0
1040,JPN,Asia,2020,Summer,262,132.0,130.0,4.0
704,GBR,Europe,1984,Summer,224,96.0,128.0,4.0
90,AUS,Oceania,2000,Summer,286,161.0,125.0,4.0
364,CHN,Asia,2004,Summer,197,87.0,110.0,4.0
253,BRA,Americas,2016,Summer,286,178.0,108.0,4.0
1023,JPN,Asia,1988,Summer,143,37.0,106.0,4.0



📊 Sustained delegation growth (3+ consecutive, Summer)


,NOC,Season,Region,Consecutive_Increases,Period_Start,Period_End,Total_Delta_pp
0,AUS,Summer,Oceania,7,1964,1988,164.0
1,ITA,Summer,Europe,6,2004,2024,69.0
2,IRL,Summer,Europe,6,1976,1996,46.0
3,ESP,Summer,Europe,6,1972,1992,222.0
4,COL,Summer,Americas,5,1988,2024,66.0
5,FRA,Summer,Europe,5,1980,1996,95.0
6,FRG,Summer,Europe,5,1972,1988,130.0
7,SWE,Summer,Europe,5,1972,1988,71.0
8,UKR,Summer,Europe,5,2000,2016,138.0
9,EGY,Summer,Africa,4,2012,2024,18.0


In [12]:
regions = ['Africa', 'Americas', 'Asia', 'Europe', 'Oceania']
paralympics_summer = paralympics[(paralympics['Season'] == 'Summer') & (paralympics['Total'] >= MIN_DELEGATION)]

fig = make_subplots(rows=3, cols=2,
                    subplot_titles=regions + ['All Regions Compared'],
                    vertical_spacing=0.08, horizontal_spacing=0.06)

for idx, region in enumerate(regions):
    row = idx // 2 + 1
    col = idx % 2 + 1
    region_data = paralympics_summer[paralympics_summer['Region'] == region]

    for noc in sorted(region_data['NOC'].unique()):
        country = region_data[region_data['NOC'] == noc].sort_values('Year')
        fig.add_trace(go.Scatter(
            x=country['Year'], y=country['Total'],
            mode='lines', name=noc,
            line=dict(color=COLORS['background_line'], width=1),
            opacity=0.4,
            hovertemplate=f'<b>{noc}</b><br>Year: %{{x}}<br>Athletes: %{{y}}<extra></extra>',
            legendgroup=region, showlegend=False), row=row, col=col)

    regional_total = (region_data.groupby('Year')['Total'].mean().reset_index())
    fig.add_trace(go.Scatter(
        x=regional_total['Year'], y=regional_total['Total'],
        mode='lines+markers', name=f'{region} avg',
        line=dict(color=COLORS[region], width=3),
        marker=dict(size=5),
        hovertemplate=f'<b>{region} average</b><br>Year: %{{x}}<br>Athletes: %{{y:.0f}}<extra></extra>',
        legendgroup='averages', showlegend=True), row=row, col=col)

for region in regions:
    region_data = paralympics_summer[paralympics_summer['Region'] == region]
    regional_total = region_data.groupby('Year')['Total'].mean().reset_index()
    fig.add_trace(go.Scatter(
        x=regional_total['Year'], y=regional_total['Total'],
        mode='lines+markers', name=region,
        line=dict(color=COLORS[region], width=3),
        marker=dict(size=5),
        hovertemplate=f'<b>{region}</b><br>Year: %{{x}}<br>Athletes: %{{y:.0f}}<extra></extra>',
        legendgroup='summary', showlegend=False), row=3, col=2)

fig.update_layout(
    height=900, width=1100,
    title_text='Paralympic Summer Games — Delegation Size by IOC Region',
    title_font_size=16,
    hovermode='closest',
    template='plotly_white')
fig.show()

##1.4. Paralympic gender equality



In [13]:
print("--- Paralympic Gender Equality (% Female) ---")
paralympics_full_gender, paralympics_changes_gender = calculate_edition_changes(
    paralympics, min_total=10)
print(f"Valid comparisons: {len(paralympics_changes_gender):,}")

paralympics_sustained_gender = find_sustained_increases(paralympics_changes_gender)
print(f"Countries with 3+ consecutive increases: {len(paralympics_sustained_gender)}")

print("\n📊 Top 15 increases in % Female (Summer)")
top_paralympics_gender = (paralympics_changes_gender[paralympics_changes_gender.Season == 'Summer']
                          .nlargest(15, 'Delta_Pct_Female')[DISPLAY_COLS_GENDER])
display(top_paralympics_gender)

print("\n📊 Sustained gender equality increases (3+ consecutive, Summer)")
paralympics_sus_gender = paralympics_sustained_gender[paralympics_sustained_gender.Season == 'Summer']
display(paralympics_sus_gender.head(15))

--- Paralympic Gender Equality (% Female) ---
Valid comparisons: 806
Countries with 3+ consecutive increases: 45

📊 Top 15 increases in % Female (Summer)


,NOC,Region,Year,Season,Female,Male,Total,Pct_Female,Prev_Pct_Female,Delta_Pct_Female,Year_Gap
1626,RWA,Africa,2016,Summer,12,1,13,92.31,0.00,92.31,4.0
812,HKG,Asia,1984,Summer,11,14,25,44.00,0.00,44.00,4.0
1665,SLO,Europe,2004,Summer,14,14,28,50.00,11.76,38.24,4.0
1252,MEX,Americas,1992,Summer,10,9,19,52.63,15.15,37.48,4.0
1543,POR,Europe,1988,Summer,6,7,13,46.15,11.76,34.39,4.0
1184,LTU,Europe,2008,Summer,14,12,26,53.85,20.00,33.85,4.0
1932,USA,Americas,1964,Summer,20,45,65,30.77,0.00,30.77,4.0
1784,SWE,Europe,2020,Summer,17,12,29,58.62,31.58,27.04,4.0
543,ESP,Europe,1972,Summer,9,11,20,45.00,18.18,26.82,4.0
1888,UAE,Asia,2012,Summer,4,11,15,26.67,0.00,26.67,8.0



📊 Sustained gender equality increases (3+ consecutive, Summer)


,NOC,Season,Region,Consecutive_Increases,Period_Start,Period_End,Total_Delta_pp
0,USA,Summer,Americas,7,1996,2020,25.2
2,ISR,Summer,Europe,5,2004,2020,48.5
3,TPE,Summer,Asia,5,2000,2016,54.4
4,FRA,Summer,Europe,5,2000,2016,17.4
5,UKR,Summer,Europe,5,2000,2016,29.2
6,GBR,Summer,Europe,4,1992,2004,15.1
8,DEN,Summer,Europe,4,1980,1992,39.3
9,BRA,Summer,Americas,4,1984,1996,23.0
11,CHN,Summer,Asia,4,2012,2024,15.2
12,AUT,Summer,Europe,4,2008,2020,18.9


In [14]:
fig = make_subplots(rows=3, cols=2,
                    subplot_titles=regions + ['All Regions Compared'],
                    vertical_spacing=0.08, horizontal_spacing=0.06)

for idx, region in enumerate(regions):
    row = idx // 2 + 1
    col = idx % 2 + 1
    region_data = paralympics_summer[paralympics_summer['Region'] == region]

    for noc in sorted(region_data['NOC'].unique()):
        country = region_data[region_data['NOC'] == noc].sort_values('Year')
        fig.add_trace(go.Scatter(
            x=country['Year'], y=country['Pct_Female'],
            mode='lines', name=noc,
            line=dict(color=COLORS['background_line'], width=1),
            opacity=0.4,
            hovertemplate=f'<b>{noc}</b><br>Year: %{{x}}<br>% Female: %{{y:.1f}}%<extra></extra>',
            legendgroup=region, showlegend=False), row=row, col=col)

    regional_avg = (region_data.groupby('Year')
                    .apply(lambda g: (g['Female'].sum() / g['Total'].sum() * 100))
                    .reset_index(name='Pct_Female'))
    fig.add_trace(go.Scatter(
        x=regional_avg['Year'], y=regional_avg['Pct_Female'],
        mode='lines+markers', name=f'{region} avg',
        line=dict(color=COLORS[region], width=3),
        marker=dict(size=5),
        hovertemplate=f'<b>{region} average</b><br>Year: %{{x}}<br>% Female: %{{y:.1f}}%<extra></extra>',
        legendgroup='averages', showlegend=True), row=row, col=col)

    fig.add_hline(y=50, line_dash="dash", line_color="gray",
                  opacity=0.4, row=row, col=col)

for region in regions:
    region_data = paralympics_summer[paralympics_summer['Region'] == region]
    regional_avg = (region_data.groupby('Year')
                    .apply(lambda g: (g['Female'].sum() / g['Total'].sum() * 100))
                    .reset_index(name='Pct_Female'))
    fig.add_trace(go.Scatter(
        x=regional_avg['Year'], y=regional_avg['Pct_Female'],
        mode='lines+markers', name=region,
        line=dict(color=COLORS[region], width=3),
        marker=dict(size=5),
        hovertemplate=f'<b>{region}</b><br>Year: %{{x}}<br>% Female: %{{y:.1f}}%<extra></extra>',
        legendgroup='summary', showlegend=False), row=3, col=2)
fig.add_hline(y=50, line_dash="dash", line_color="gray", opacity=0.4, row=3, col=2)

fig.update_yaxes(range=[0, 105], ticksuffix='%')
fig.update_layout(
    height=900, width=1100,
    title_text='Paralympic Summer Games — Female Athlete Participation by IOC Region',
    title_font_size=16,
    hovermode='closest',
    template='plotly_white')
fig.show()

#2. Selected cases

In [15]:
CASES = {
    'Olympic Gender Equality': {
        'ALG': '7 consecutive increases (1988–2012), +44.7pp',
        'BRA': '8 consecutive increases (1976–2004), +45.3pp',
        'MAS': '7 consecutive increases (1996–2020), +60.0pp'},
    'Paralympic Participation': {
        'KOR': 'Seoul 1988: 18 → 226 athletes (+1,155%)',
        'ESP': '6 consecutive increases (1972–1992), 11 → 233',
        'UKR': '5 consecutive increases (2000–2016), 30 → 168'},
    'Paralympic Gender (Intersectional)': {
        'RWA': '0% → 92.3% female in one edition (2012–2016)',
        'SLO': '4 consecutive increases (2000–2012), +61.0pp',
        'TPE': '5 consecutive increases (2000–2016), +54.4pp'}}

for dimension, cases in CASES.items():
    print(f"\n{dimension}:")
    for noc, note in cases.items():
        print(f"  • {noc}: {note}")


Olympic Gender Equality:
  • ALG: 7 consecutive increases (1988–2012), +44.7pp
  • BRA: 8 consecutive increases (1976–2004), +45.3pp
  • MAS: 7 consecutive increases (1996–2020), +60.0pp

Paralympic Participation:
  • KOR: Seoul 1988: 18 → 226 athletes (+1,155%)
  • ESP: 6 consecutive increases (1972–1992), 11 → 233
  • UKR: 5 consecutive increases (2000–2016), 30 → 168

Paralympic Gender (Intersectional):
  • RWA: 0% → 92.3% female in one edition (2012–2016)
  • SLO: 4 consecutive increases (2000–2012), +61.0pp
  • TPE: 5 consecutive increases (2000–2016), +54.4pp


##2.1. Gender equality

In [16]:
selected_cases = {
    'ALG': {'label': 'Algeria', 'region': 'Africa', 'period': (1988, 2012)},
    'BRA': {'label': 'Brazil', 'region': 'Americas', 'period': (1976, 2004)},
    'MAS': {'label': 'Malaysia', 'region': 'Asia', 'period': (1996, 2020)}}

fig = make_subplots(rows=2, cols=2,
                    subplot_titles=[
                        'Algeria (Africa)', 'Brazil (Americas)',
                        'Malaysia (Asia)', 'Why these countries?'],
                    vertical_spacing=0.12, horizontal_spacing=0.06)

for idx, (noc, info) in enumerate(selected_cases.items()):
    row = idx // 2 + 1
    col = idx % 2 + 1
    region = info['region']
    region_data = olympics_summer[olympics_summer['Region'] == region]
    start, end = info['period']

    # Sustained period shading
    fig.add_vrect(x0=start - 1, x1=end + 1,
                  fillcolor=COLORS['highlight'], opacity=0.06,
                  line_width=0, row=row, col=col)

    # Background countries (clickable)
    for other_noc in sorted(region_data['NOC'].unique()):
        if other_noc == noc:
            continue
        other = region_data[region_data['NOC'] == other_noc].sort_values('Year')
        fig.add_trace(go.Scatter(
            x=other['Year'], y=other['Pct_Female'],
            mode='lines', name=other_noc,
            line=dict(color=COLORS['background_line'], width=0.8),
            opacity=0.3,
            hovertemplate=f'<b>{other_noc}</b><br>Year: %{{x}}<br>% Female: %{{y:.1f}}%<extra></extra>',
            showlegend=False), row=row, col=col)

    # Highlighted country
    country = region_data[region_data['NOC'] == noc].sort_values('Year')
    fig.add_trace(go.Scatter(
        x=country['Year'], y=country['Pct_Female'],
        mode='lines+markers+text', name=info['label'],
        line=dict(color=COLORS['highlight'], width=3),
        marker=dict(size=7),
        text=[f"{v:.1f}%" for v in country['Pct_Female']],
        textposition='top center', textfont=dict(size=8, color=COLORS['highlight']),
        hovertemplate=f'<b>{info["label"]}</b><br>Year: %{{x}}<br>% Female: %{{y:.1f}}%<extra></extra>',
        showlegend=True), row=row, col=col)

    # Parity line
    fig.add_hline(y=50, line_dash="dash", line_color="gray", opacity=0.4, row=row, col=col)

# Bottom-right: methodology bar chart
top10 = (olympics_sustained[olympics_sustained.Season == 'Summer']
         .head(10).sort_values('Total_Delta_pp'))

bar_colors = [COLORS['highlight'] if noc in selected_cases else COLORS['male']
              for noc in top10['NOC']]
bar_text = [f"{int(row['Period_Start'])}–{int(row['Period_End'])} ({int(row['Consecutive_Increases'])} ed.)"
            for _, row in top10.iterrows()]

fig.add_trace(go.Bar(
    y=top10['NOC'], x=top10['Total_Delta_pp'],
    orientation='h', marker_color=bar_colors,
    text=bar_text, textposition='outside', textfont=dict(size=9),
    hovertemplate='<b>%{y}</b><br>Total increase: %{x:.1f}pp<extra></extra>',
    showlegend=False), row=2, col=2)

fig.update_yaxes(range=[0, 105], ticksuffix='%', row=1, col=1)
fig.update_yaxes(range=[0, 105], ticksuffix='%', row=1, col=2)
fig.update_yaxes(range=[0, 105], ticksuffix='%', row=2, col=1)
fig.update_xaxes(title_text='Total increase in % Female (pp)', row=2, col=2)

fig.update_layout(
    height=800, width=1100,
    title_text='Olympic Gender Equality — Selected Case Studies',
    title_font_size=16,
    hovermode='closest',
    template='plotly_white')
fig.show()

###2.1.1 Algeria

Algeria is an interesting political and social case to analyse, as it is one of the few Arab countries marked by a strong feminist movement (in the 1980s). As a result of this movement, a new constitution was approved in 1989 that included gender equality as a constitutional principle, although the movement was already producing effects before the reform: the increase in female participation in the Olympic delegation began in 1988. However, the constitution did not remove the cultural and religious barriers to women's sports participation: Islamic modesty norms made it difficult to wear sport-specific clothing, and women lacked access to clubs and institutions. Faced with these limitations, some women chose to pursue sports that did not necessarily require major infrastructure, and this is how women's athletics was strongly boosted. One example is middle-distance runner Hassiba Boulmerka, Olympic gold medallist in Barcelona 1992 and two-time world champion, who started out running in the streets of her city and became a symbol of resistance for Algerian women. A contrast between religious-cultural constraints and a powerful feminist movement demanding change. Between 1988 and 2012, the percentage of women in Algeria's Olympic delegation grew steadily over seven consecutive editions, accumulating an increase of 44.7 percentage points.

###2.1.2. Brazil

Between 1976 and 2004, the percentage of women in Brazil's Olympic delegation grew steadily over eight consecutive editions, accumulating an increase of 45.3 percentage points. During the years when the growth in Brazil's female participation at the Olympic Games began, Decree-Law 3,199 of 1941 was in force, prohibiting women from participating in sports "incompatible with their nature": 38 years of prohibition, the only one imposed by state decree in the world. However, Brazil is one of the great sporting powers, characterised primarily by the country's sporting culture: it is common to see children playing in the streets, on beaches, practising sports in squares and parks, before ever joining a formal sports club. For decades, that street culture was off-limits for women, but in practice the law was not always strictly enforced, in fact, the growth in female Olympic participation began in 1976, three years before the formal repeal of the decree. After the ban was lifted in 1979, still under military dictatorship, which would not end until 1985, girls and women began to practise sport freely, and in 1981 the first women's football league was created. Brazilian women's sport stands out primarily in team sports, like football or volleyball, but also in disciplines such as artistic gymnastics and athletics.

###2.1.3. Malaysia

Malaysia is another country that faced a duality between a traditional Muslim culture and a growing modernisation and interculturality. The 60 percentage-point increase in female participation between 1996 and 2020 is consistent with national policies implemented years earlier. On one hand, an education policy developed from the 1980s onwards expanded access to higher education for women, which may have facilitated their entry into university sport. On the other hand, key sports infrastructure was established in the 1990s: the National Sports Council as a governmental institution for sports development, and the Bukit Jalil Sports School as an elite training centre, both with programmes specifically designed for women's sport. Athletes such as Nicol David, the longest-reigning world squash champion, or Pandelela Rinong, the first Malaysian woman to win an Olympic medal, at London 2012, are products of that ecosystem and at the same time role models who fuelled the participation of new generations. Between 1996 and 2020, the percentage of women in Malaysia's Olympic delegation grew steadily over seven consecutive editions.

##2.2. Paralympic participation

In [17]:
selected_para = {
    'KOR': {'label': 'South Korea', 'region': 'Asia', 'period': (1984, 1988)},
    'ESP': {'label': 'Spain', 'region': 'Europe', 'period': (1972, 1992)},
    'UKR': {'label': 'Ukraine', 'region': 'Europe', 'period': (2000, 2016)}}

fig = make_subplots(rows=2, cols=2,
                    subplot_titles=[
                        'South Korea (Asia) — Seoul 1988 host effect',
                        'Spain (Europe) — Barcelona 1992 as culmination',
                        'Ukraine (Europe) — Post-independence growth',
                        'Why these countries?'],
                    vertical_spacing=0.12, horizontal_spacing=0.06)

for idx, (noc, info) in enumerate(selected_para.items()):
    row = idx // 2 + 1
    col = idx % 2 + 1
    region = info['region']
    region_data = paralympics_summer[paralympics_summer['Region'] == region]
    start, end = info['period']

    fig.add_vrect(x0=start - 1, x1=end + 1,
                  fillcolor=COLORS['highlight'], opacity=0.06,
                  line_width=0, row=row, col=col)

    for other_noc in sorted(region_data['NOC'].unique()):
        if other_noc == noc:
            continue
        other = region_data[region_data['NOC'] == other_noc].sort_values('Year')
        fig.add_trace(go.Scatter(
            x=other['Year'], y=other['Total'],
            mode='lines', name=other_noc,
            line=dict(color=COLORS['background_line'], width=0.8),
            opacity=0.3,
            hovertemplate=f'<b>{other_noc}</b><br>Year: %{{x}}<br>Athletes: %{{y}}<extra></extra>',
            showlegend=False), row=row, col=col)

    country = region_data[region_data['NOC'] == noc].sort_values('Year')
    fig.add_trace(go.Scatter(
        x=country['Year'], y=country['Total'],
        mode='lines+markers+text', name=info['label'],
        line=dict(color=COLORS['highlight'], width=3),
        marker=dict(size=7),
        text=[str(int(v)) for v in country['Total']],
        textposition='top center', textfont=dict(size=8, color=COLORS['highlight']),
        hovertemplate=f'<b>{info["label"]}</b><br>Year: %{{x}}<br>Athletes: %{{y}}<extra></extra>',
        showlegend=True), row=row, col=col)

# Bottom-right: methodology bar chart
top10_para = (paralympics_sustained_total[paralympics_sustained_total.Season == 'Summer']
              .head(10).sort_values('Total_Delta_pp'))

bar_colors = [COLORS['highlight'] if noc in selected_para else COLORS['male']
              for noc in top10_para['NOC']]
bar_text = [f"{int(row['Period_Start'])}–{int(row['Period_End'])} ({int(row['Consecutive_Increases'])} ed.)"
            for _, row in top10_para.iterrows()]

fig.add_trace(go.Bar(
    y=top10_para['NOC'], x=top10_para['Total_Delta_pp'],
    orientation='h', marker_color=bar_colors,
    text=bar_text, textposition='outside', textfont=dict(size=9),
    hovertemplate='<b>%{y}</b><br>Total growth: %{x:.0f} athletes<extra></extra>',
    showlegend=False), row=2, col=2)

fig.update_xaxes(title_text='Total delegation growth (athletes)', row=2, col=2)

fig.update_layout(
    height=800, width=1100,
    title_text='Paralympic Participation — Selected Case Studies',
    title_font_size=16,
    hovermode='closest',
    template='plotly_white')
fig.show()

###2.2.1. South Korea
The case of South Korea appeals to me because it is closely linked to social pressure. The analysis begins in 1981, with the enactment of the Welfare of Persons with Disabilities Act, a law pushed by dictator Chun Doo-hwan to legitimise his military coup, not out of conviction. Following the approval of this law, the very same government forced the confinement of people with disabilities in institutions to remove them from the streets ahead of the 1986 Asian Games and the 1988 Paralympic Games (where they were also the host nation), with the aim of "protecting the country's image". Many of those confined suffered violence and forced labour. In response, the Disability Rights Movement emerged during 1987 and 1988, seeking to boycott the Games and demanding legislation that would genuinely protect people with disabilities, a goal finally achieved in 1990 with the Employment Promotion for Persons with Disabilities Act. The massive 1,155% surge in the Paralympic delegation in 1988 (from 18 to 226 athletes) was therefore more of an image operation than a real commitment to inclusion, and not a result of the country's social or sports policies. What came afterwards is the truly interesting part: social pressure to reverse this situation led to a decline in the delegation, partly also because the automatic host-nation quota disappeared, but undoubtedly an improvement in athletes' quality of life, visibility and dignity. After the post-1988 drop, South Korea's Paralympic delegation stabilised at around 80–90 athletes over the following decades: the infrastructure remained, even though the artificial inflation was gone.


###2.2.2. Spain

Between 1972 and 1992, Spain's Paralympic delegation grew steadily over six consecutive editions, from 11 to 233 athletes. The case of Spain is also marked by the home advantage of 1992, but its growth did not start there: Spain's Paralympic delegation had been growing steadily for 20 years before Barcelona, making the host effect the culmination rather than the origin. The first milestone was the creation of the first Spanish organisation for disability sport, under the presidency of Juan Antonio Samaranch at the Spanish Olympic Committee in 1968, along with Spain's first participation at the Paralympic Games. From 1986 onwards, the ONCE Foundation provided funding and organisational structure to strengthen Paralympic sport in the country, including the 1992 Paralympic Games. The Sports Act of 1990 created four new adapted sport federations, and the Barcelona Paralympic Games offered free admission and were broadcast live for the first time in history, seeking to foster interest in adapted sport. Unlike South Korea, the momentum gained at Barcelona 92 continued after the competition ended, and a few years later the Spanish Paralympic Committee was established (1995) along with the ADOP Plan, a public-private funding model for Paralympic athletes.



###2.2.3. Ukrain

The growth of Paralympic sport in Ukraine is the only case with no connection whatsoever to a home Paralympic Games. Here, one actor stands out: Valerii Sushkevych, a former Paralympic swimmer and member of parliament. Before reaching that position, when Ukraine was still part of the USSR, a Soviet official told him that "disabled people don't do sport, they belong in hospitals." He reversed that vision not only by dedicating his life to adapted sport, but by leading the Ukrainian Paralympic movement from 1992 and establishing, under the Invasport programme, a Paralympic sports school in every region of Ukraine: schools that did not only train elite athletes, but also invited children with disabilities to rehabilitation and physical activity in order to identify talent from the grassroots up. The growth in Paralympic participation from that point is striking: 30 athletes at their debut as an independent country in 1996, 67 in 2000, 90 in 2004, and 168 in 2016. At Sydney 2000, only their second participation, the Paralympic team won 37 medals, a historic milestone for the country, which translated into significant political and social impact: the team was welcomed as heroes at the Presidential Palace, more resources were allocated to Paralympic sport, and in 2002 the National Paralympic High Performance Centre was built.



##2.3. Paralympic gender equality

In [18]:
selected_inter = {
    'RWA': {'label': 'Rwanda', 'region': 'Africa', 'period': (2012, 2016)},
    'SLO': {'label': 'Slovenia', 'region': 'Europe', 'period': (2000, 2012)},
    'TPE': {'label': 'Taiwan (Chinese Taipei)', 'region': 'Asia', 'period': (2000, 2016)}}

fig = make_subplots(rows=2, cols=2,
                    subplot_titles=[
                        'Rwanda (Africa) — 0% → 92% in one edition',
                        'Slovenia (Europe) — 4 consecutive increases',
                        'Taiwan (Asia) — 5 consecutive increases',
                        'Why these countries?'],
                    vertical_spacing=0.12, horizontal_spacing=0.06)

for idx, (noc, info) in enumerate(selected_inter.items()):
    row = idx // 2 + 1
    col = idx % 2 + 1
    region = info['region']
    region_data = paralympics_summer[paralympics_summer['Region'] == region]
    start, end = info['period']

    fig.add_vrect(x0=start - 1, x1=end + 1,
                  fillcolor=COLORS['highlight'], opacity=0.06,
                  line_width=0, row=row, col=col)

    for other_noc in sorted(region_data['NOC'].unique()):
        if other_noc == noc:
            continue
        other = region_data[region_data['NOC'] == other_noc].sort_values('Year')
        fig.add_trace(go.Scatter(
            x=other['Year'], y=other['Pct_Female'],
            mode='lines', name=other_noc,
            line=dict(color=COLORS['background_line'], width=0.8),
            opacity=0.3,
            hovertemplate=f'<b>{other_noc}</b><br>Year: %{{x}}<br>% Female: %{{y:.1f}}%<extra></extra>',
            showlegend=False), row=row, col=col)

    country = region_data[region_data['NOC'] == noc].sort_values('Year')
    fig.add_trace(go.Scatter(
        x=country['Year'], y=country['Pct_Female'],
        mode='lines+markers+text', name=info['label'],
        line=dict(color=COLORS['highlight'], width=3),
        marker=dict(size=7),
        text=[f"{v:.1f}%" for v in country['Pct_Female']],
        textposition='top center', textfont=dict(size=8, color=COLORS['highlight']),
        hovertemplate=f'<b>{info["label"]}</b><br>Year: %{{x}}<br>% Female: %{{y:.1f}}%<extra></extra>',
        showlegend=True), row=row, col=col)

    fig.add_hline(y=50, line_dash="dash", line_color="gray", opacity=0.4, row=row, col=col)

# Bottom-right: methodology bar chart
top10_inter = (paralympics_sustained_gender[paralympics_sustained_gender.Season == 'Summer']
               .head(10).sort_values('Total_Delta_pp'))

bar_colors = [COLORS['highlight'] if noc in selected_inter else COLORS['male']
              for noc in top10_inter['NOC']]
bar_text = [f"{int(row['Period_Start'])}–{int(row['Period_End'])} ({int(row['Consecutive_Increases'])} ed.)"
            for _, row in top10_inter.iterrows()]

fig.add_trace(go.Bar(
    y=top10_inter['NOC'], x=top10_inter['Total_Delta_pp'],
    orientation='h', marker_color=bar_colors,
    text=bar_text, textposition='outside', textfont=dict(size=9),
    hovertemplate='<b>%{y}</b><br>Total increase: %{x:.1f}pp<extra></extra>',
    showlegend=False), row=2, col=2)

fig.update_yaxes(range=[0, 105], ticksuffix='%', row=1, col=1)
fig.update_yaxes(range=[0, 105], ticksuffix='%', row=1, col=2)
fig.update_yaxes(range=[0, 105], ticksuffix='%', row=2, col=1)
fig.update_xaxes(title_text='Total increase in % Female (pp)', row=2, col=2)

fig.update_layout(
    height=800, width=1100,
    title_text='Paralympic Gender Equality (Intersectional) — Selected Case Studies',
    title_font_size=16,
    hovermode='closest',
    template='plotly_white')
fig.show()

#3. Conclusion

None of the six cases analysed can be explained by a single factor. In all of them, sustained growth was the product of the intersection of at least three elements: a political-institutional change (a new constitution, a law, a democratic transition, independence), an investment in sports infrastructure (schools, federations, funding programmes), and a human or symbolic catalyst (a social movement, a landmark athlete, a visionary leader, or a major sporting event). Laws alone did not generate participation. Events alone inflated numbers that later deflated. Individuals alone could not scale without structure. It was always the combination of the three that produced lasting change. The contrast between South Korea and Spain illustrates this clearly: both hosted Paralympic Games, but only in Spain did the growth endure, because it had two decades of institutional building behind it. Data does not measure intentions or political will; it measures outcomes. And the outcomes show that structural change in sports participation, whether in gender or inclusion, is not decreed: it is built.